# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)



## 1. Two paper findings + my methodology questions

### Finding: "The Content Performance Curve" (Finding #2)

**The claim:** Health score follows a lifecycle — peaking at 61-90 days (33.1), declining
through 271-365 days (14.0), with a "recovery" at 365+ days (25.1) that the paper
attributes to refreshed older content.

**My methodology question:** The 365+ "recovery" bucket mixes two very different
populations — pages that were refreshed and pages that weren't — without splitting them
apart in this specific chart. The paper itself later shows (in the Age-Freshness Matrix)
that refreshed 365+ pages score dramatically higher than unrefreshed ones. My question,
asked the way I'd want it asked of my own work: **is the 365+ health-score-by-age curve
in Finding #2 itself confounded by refresh status, and if the unrefreshed 365+ pages were
isolated, would the "recovery" still appear, or would it flatten into continued decline?**
This isn't a request to redo the whole paper — the Age-Freshness Matrix later on already
answers this concern well — but as a standalone chart, Finding #2's age-only curve risks a
reader assuming age alone explains the recovery, when the paper's own later section shows
freshness is the real driver. A one-line pointer earlier ("see the Age-Freshness Matrix for
the refresh-controlled view") would close this gap cleanly.

### Finding: ML Appendix — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

**The claim:** A logistic regression identifies which features separate growing from
declining pages, with content age as the strongest *negative* signal and days-visible/
recent impressions as the strongest positive signals; the paper reports 71% holdout
accuracy.

**My methodology question:** The paper is careful elsewhere to flag when a target is
partly constructed from its own inputs (it does this explicitly for the Random Forest →
health score model, noting importance is "descriptive rather than causal" because health
score includes position and impressions as components). The growth/decline logistic
regression doesn't get the same explicit caveat. My question, in the same constructive
spirit: **how was "growing" vs. "declining" defined for this label, and does that
definition overlap with any of the input features (e.g., if growth is defined from a
30-day impression trend, and "recent impressions" is also a top feature, is the model
partly predicting itself)?** 71% accuracy is a modest, believable number — not suspiciously
perfect — so this likely isn't a serious leakage problem, but the paper's own standard
(caveat health-score-model results for input/target overlap) would be good to apply here
too, even briefly, so a reader doesn't need to reverse-engineer whether growth and
impressions are defined independently.

In [1]:
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 148 (delta 57), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.88 MiB | 9.50 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/FlyRank-Internship


In [2]:
import duckdb
from google.colab import userdata
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score, precision_score

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [3]:
model_data = con.sql("""
    WITH march_data AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    ),
    joined AS (
        SELECT
            m.*,
            c.word_count,
            c.search_volume,
            c.competition,
            c.cpc
        FROM march_data m
        JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
            ON m.content_hash_id = c.content_hash_id
    )
    SELECT *,
        clicks_march * 1.0 / NULLIF(impressions_march, 0) AS actual_ctr,
        CASE
            WHEN avg_position_march <= 3 THEN '1. position 1-3'
            WHEN avg_position_march <= 10 THEN '2. position 4-10'
            WHEN avg_position_march <= 20 THEN '3. position 11-20'
            ELSE '4. position 20+'
        END AS position_bucket
    FROM joined
    WHERE impressions_march >= 500 AND avg_position_march > 0 AND avg_position_march <= 20
""").df()

bucket_expected = {'1. position 1-3': 0.003765, '2. position 4-10': 0.003208, '3. position 11-20': 0.002625}
model_data['expected_ctr'] = model_data['position_bucket'].map(bucket_expected)
model_data['label'] = (model_data['actual_ctr'] < model_data['expected_ctr'] * 0.7).astype(int)

numeric_features = ['impressions_march', 'avg_position_march', 'word_count', 'search_volume', 'competition', 'cpc']
model_data_clean = model_data.dropna(subset=numeric_features + ['label']).copy()

print("Total clean rows:", model_data_clean.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total clean rows: (39204, 13)


## 2. My model under an honest split (before/after)

**Before (plain random 70/30 split, no client grouping):** ROC AUC = 0.566, Precision =
0.545

**After (grouped by client, 0 client overlap confirmed):** ROC AUC = 0.532, Precision =
0.538

**What changed and why:** Removing the client-grouping safeguard inflates ROC AUC by about
0.034 points (0.566 vs. 0.532) and precision by about 0.008 points. This is the expected
direction and size for client leakage: when the same client's pages can appear in both
train and test, the model can partly learn client-specific quirks (e.g. a particular site's
typical position range, content style, or CTR baseline) rather than a pattern that
generalizes to genuinely new clients. The gap here is modest, not dramatic — this signal
set doesn't leak heavily through client identity, but it does leak some, which is exactly
why the grouped design (Week 5's original approach) is the more honest number to report
and trust going forward, even though it looks slightly less impressive.

This confirms the split design decision made in Week 5 was the right one — the "after"
number (0.532 AUC) is the one that should be quoted in any public-safe summary of this
model's performance, not the inflated 0.566.

In [4]:
# BEFORE: plain random split (no client grouping) — the naive, less honest approach
X = model_data_clean[numeric_features]
y = model_data_clean['label']

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler_before = StandardScaler()
X_train_random_scaled = scaler_before.fit_transform(X_train_random)
X_test_random_scaled = scaler_before.transform(X_test_random)

model_before = LogisticRegression(max_iter=1000, random_state=42)
model_before.fit(X_train_random_scaled, y_train_random)

pred_proba_before = model_before.predict_proba(X_test_random_scaled)[:, 1]
pred_label_before = model_before.predict(X_test_random_scaled)

auc_before = roc_auc_score(y_test_random, pred_proba_before)
precision_before = precision_score(y_test_random, pred_label_before)

print(f"BEFORE (plain random split) — ROC AUC: {auc_before:.3f}, Precision: {precision_before:.3f}")


# AFTER: grouped split by client (the honest design from Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_data_clean, groups=model_data_clean['client_hash_id']))

train_grouped = model_data_clean.iloc[train_idx].reset_index(drop=True)
test_grouped = model_data_clean.iloc[test_idx].reset_index(drop=True)

# Confirm zero client overlap
overlap = set(train_grouped['client_hash_id']) & set(test_grouped['client_hash_id'])
print("Client overlap (should be 0):", len(overlap))

X_train_grouped = train_grouped[numeric_features]
y_train_grouped = train_grouped['label']
X_test_grouped = test_grouped[numeric_features]
y_test_grouped = test_grouped['label']

scaler_after = StandardScaler()
X_train_grouped_scaled = scaler_after.fit_transform(X_train_grouped)
X_test_grouped_scaled = scaler_after.transform(X_test_grouped)

model_after = LogisticRegression(max_iter=1000, random_state=42)
model_after.fit(X_train_grouped_scaled, y_train_grouped)

pred_proba_after = model_after.predict_proba(X_test_grouped_scaled)[:, 1]
pred_label_after = model_after.predict(X_test_grouped_scaled)

auc_after = roc_auc_score(y_test_grouped, pred_proba_after)
precision_after = precision_score(y_test_grouped, pred_label_after)

print(f"AFTER (grouped by client) — ROC AUC: {auc_after:.3f}, Precision: {precision_after:.3f}")

# Comparison table
comparison_df = pd.DataFrame({
    'Split design': ['BEFORE: plain random split', 'AFTER: grouped by client'],
    'ROC AUC': [auc_before, auc_after],
    'Precision': [precision_before, precision_after]
})
print("\n", comparison_df)

BEFORE (plain random split) — ROC AUC: 0.566, Precision: 0.545
Client overlap (should be 0): 0
AFTER (grouped by client) — ROC AUC: 0.532, Precision: 0.538

                  Split design   ROC AUC  Precision
0  BEFORE: plain random split  0.566118   0.545299
1    AFTER: grouped by client  0.531652   0.537646


## 3. Leakage audit

Running the same leakage checklist from Week 3's data contract, applied to my final
feature set (`impressions_march`, `avg_position_march`, `word_count`, `search_volume`,
`competition`, `cpc`) and label (`actual_ctr < expected_ctr * 0.7`).

**Checklist, applied honestly:**

1. **Are any features calculated after the decision point?** No — all six features come
   from the same March 2026 window as the label. No feature uses April data or later.

2. **Does the feature window overlap the target window?** Yes, and this is a real,
   acknowledged limitation, not a leakage bug in the strict sense: the label is defined
   from `actual_ctr`, which is itself computed from `impressions_march` and
   `clicks_march` — the same window used for the `impressions_march` feature. This isn't
   future-leakage (nothing from outside March sneaks in), but it does mean the label and
   at least one feature are drawn from the same snapshot, which is why this model should
   be read as *describing* March, not *predicting* a future outcome. A stronger future
   version of this lane would need a genuine prior-window-predicts-next-window design
   (as the Week 3 lane guide recommends), which this baseline does not yet attempt.

3. **If I rebuilt any product output, did it slip in as a normal feature?** No FlyRank
   product fields (`health_score`, `priority_score`, `action_type`, any optimization
   flag) were used anywhere — the release doesn't ship them, and none were reconstructed.

4. **Does a derived field secretly encode the target?** This is the most serious finding
   from Week 5, restated here formally: my original "baseline comparison"
   (`actual_ctr < expected_ctr`) was discovered to be nearly the same condition as the
   label itself (`actual_ctr < expected_ctr * 0.7`), producing a circular, near-perfect
   ROC AUC (0.997) that did not reflect genuine predictive skill. This was caught and
   excluded from any final claim — see Week 5's writeup for the full account. The
   Logistic Regression's actual input features (position, impressions, word count,
   search volume, competition, CPC) do not have this problem; none of them are
   arithmetically derived from the label.

5. **Are duplicate or related rows split across train and test in a way that makes the
   test too easy?** Addressed directly in Section 2: a plain random split allows the
   same client's pages to appear in both train and test, inflating ROC AUC by about
   0.034 points. The grouped-by-client split (0 confirmed overlap) removes this and is
   the design used for all reported "after" numbers.

6. **Am I testing on clients or time periods the model has not effectively already
   seen?** Yes, for clients (grouped split, confirmed 0 overlap) — but no, for time: all
   data comes from a single month (March 2026), so this model has not yet been tested on
   a genuinely future time period. This mirrors the Week 3 data contract's own named
   limitation (single-month slice) and remains an open gap, not a hidden one.

**One additional finding, carried from Week 3:** `content_updated_date` in `dim_content`
was found to contain negative "days since update" values relative to March (implying the
field reflects a later export-time state, not March-era history). This field was
excluded from the final feature set entirely for exactly this reason — it never entered
the model, so it does not affect the numbers reported here, but it remains a real data
quality issue worth flagging for anyone extending this work with staleness-based
features.

**Summary:** no future-window leakage and no product-flag leakage were found in the
final model. Two real, disclosed issues remain: (1) feature/label window overlap within
the same month, meaning this model describes March rather than predicts forward, and
(2) client-grouping was necessary and does measurably matter, though the effect size is
modest (~0.034 AUC).

## 4. Claim rewrite

**Original claim (Week 4, rule description):**
"These pages are already earning visibility — the gap is in conversion from impression to
click, which usually points to a fixable title or meta description problem rather than a
ranking problem."

**What's wrong with it:** This states a causal explanation ("points to a fixable title or
meta description problem") as the likely default, but my own top-10 review from the same
week found the opposite pattern in every single flagged row: pages with meaningful
impressions and zero clicks — an extreme case where a technical or indexing problem is at
least as plausible as a copy problem, as I noted explicitly at the time. The original
sentence oversells what the rule can actually tell a reviewer.

**Rewritten, safe version:**
"These pages show measurable search visibility alongside a click-through rate below what
their position would predict. This gap is observed, not diagnosed — it may reflect a
title or meta description that could be improved, but it could equally reflect a
technical, indexing, or tracking issue, especially for pages with zero clicks despite high
impressions. The rule flags where to look first; it does not identify the cause. A human
reviewer should manually check the page before assuming a content or copy fix will help."

**Why the rewrite is better:** It keeps the genuinely useful part (the rule correctly
identifies a real, statistically supported gap — Signal Check 2 confirmed CTR-vs-position
is a real relationship) while removing the unsupported causal leap. It also explicitly
routes back to the finding from my own top-10 review, instead of contradicting it. This
is decision-support language, not diagnostic language — exactly the distinction the
lane guide asks for.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.